# Mouse Healthy vs AKI Proximal-Tubule Pseudospace Comparison

This notebook compares proximal-tubule expression trajectories along the shared upstream
Scanpy DPT coordinate. Penalized GAMs estimate smooth trajectories, while blocked
permutations test whether genes and pathway scores vary significantly along pseudospace
within each condition.

Because there are only two biological samples per condition, healthy-versus-AKI
differences are descriptive and are ranked by fitted-curve effect size. Additional independent
samples are required for adequately powered between-condition inference.

Pathway overrepresentation uses a predefined tested-gene background. Enrichment p-values
and FDR identify statistically significant pathways; enrichment fold and fitted trajectory
effect sizes rank them.

## P-value guide

| Output column | Question tested | Use |
|---|---|---|
| `*_dynamic_blocked_permutation_pvalue` / `*_fdr` | Does the gene or pathway score vary along pseudospace within that condition? | Use FDR to identify statistically dynamic trajectories, then rank by effect size. |
| `enrichment_overrepresentation_pvalue` / `enrichment_overrepresentation_fdr` | Is a pathway overrepresented in a supplied statistically dynamic gene list? | Use FDR to identify statistically significant pathways, then rank by enrichment fold or trajectory effect size. |

P-values determine statistical significance; effect sizes determine ranking and biological
prioritization. Cross-condition effect sizes are descriptive because the available
biological replication is insufficient for a powered healthy-versus-AKI significance test.


## Input data, cohort definition, and shared pseudospace

The cross-condition comparison is performed using
`data/pseudspace_analysis/all_tubules_scanpy_dpt.h5ad`, the canonical Scanpy diffusion
pseudotime (DPT) output generated upstream. This object contains 45,753 tubules and
9,610 one-to-one ortholog genes retained after the upstream expression-count filter. The
analysis uses `layers["lognorm"]`, which contains library-size-normalized and
log-transformed expression values. Genes removed by the upstream filter are not
reintroduced.

The canonical DPT output contains both `total_scanpy_dpt` and
`broad_tubule_marker_call`, so the comparison has a single upstream input contract.

The analysis is restricted to tubules with `broad_tubule_marker_call == "PT"`. Healthy
tubules are further restricted to `condition == "Control"` so that ischemia-reperfusion
injury is not confounded with condition. AKI tubules are retained from the available
aki samples. The comparison therefore evaluates healthy mouse controls against mouse AKI
proximal tubules.

The saved object does not contain a separately reconstructed PT-only DPT coordinate.
The harmonized `total_scanpy_dpt` coordinate is already normalized to `[0, 1]` upstream
and is used directly:

`shared pseudospace = total_scanpy_dpt`.

No within-condition percentile reranking, rescaling, or realignment is performed. Healthy
and AKI PT tubules are modeled at their original positions on the same harmonized DPT
axis. Tubule-level values on this shared coordinate are used directly for trajectory
modeling.


In [ ]:
from pathlib import Path
import json
import os
import warnings
from concurrent.futures import ThreadPoolExecutor

CACHE_ROOT = Path('/tmp/mouse_healthy_vs_aki_pseudospace_cache')
CACHE_ROOT.mkdir(exist_ok=True)
(CACHE_ROOT / 'matplotlib').mkdir(exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(CACHE_ROOT / 'matplotlib'))
os.environ.setdefault('XDG_CACHE_HOME', str(CACHE_ROOT))

import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
import torch
from scipy.ndimage import gaussian_filter1d
from scipy.stats import spearmanr
from patsy import dmatrix
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 180

PROJECT_DIR = Path.cwd().resolve()
INPUT_PATH = PROJECT_DIR / 'data' / 'pseudspace_analysis_caleb' / 'all_tubules_scanpy_dpt.h5ad'
OUTPUT_DIR = PROJECT_DIR / 'data' / 'mouse_healthy_vs_aki_caleb'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'pseudotime_col': 'total_scanpy_dpt',
    'n_bins': 30,
    'gam_basis_df': 6,
    'gam_alpha': 1.0,
    'grid_points': 101,
    'min_detected_fraction': 0.03,
    'min_mean_expression': 0.01,
    'dynamic_permutations': 10000,
    'permutation_workers': 12,
    'permutation_batch_size': 16,
    'permutation_backend': 'cuda',
    'permutation_gpu_ids': [0],
    'permutation_gpu_batch_size': 64,
    'permutation_seed': 7,
    'dynamic_fdr': 0.05,
    'similar_curve_correlation': 0.70,
    'peak_shift_threshold': 0.15,
    'amplitude_ratio_threshold': 1.50,
    'heatmap_genes': 60,
    'pathway_min_genes': 10,
    'pathway_max_genes': 150,
    'pathway_top_plot': 12,
    'pathway_model_libraries': [
        'MSigDB_Hallmark_2020',
        'KEGG_2019_Mouse',
        'Reactome_2022',
    ],
    'gene_top_plot': 12,
    'enrichment_fdr': 0.05,
    'enrichment_priority_classes': [
        'dynamic_both_similar_shape_genes',
        'dynamic_detected_only_healthy',
        'dynamic_detected_only_aki',
    ],
}

adata = ad.read_h5ad(INPUT_PATH)
if 'lognorm' not in adata.layers:
    raise KeyError(f'{INPUT_PATH.name} is missing required lognorm layer')
required_obs = {'species', 'condition', 'broad_tubule_marker_call', CONFIG['pseudotime_col']}
missing_obs = sorted(required_obs - set(adata.obs.columns))
if missing_obs:
    raise KeyError(f'Canonical AnnData is missing required obs columns: {missing_obs}')

obs = adata.obs
pt_mask = obs['broad_tubule_marker_call'].astype(str).eq('PT').to_numpy()
species_lower = obs['species'].astype(str).str.lower()
condition_lower = obs['condition'].astype(str).str.lower()
valid_condition = ~condition_lower.isin(['', 'nan', 'none'])
mouse_mask = species_lower.eq('mouse').to_numpy()
healthy_mouse = (mouse_mask & condition_lower.eq('control').to_numpy())
aki_mouse = (mouse_mask & valid_condition.to_numpy() & ~condition_lower.eq('control').to_numpy())
keep = pt_mask & (healthy_mouse | aki_mouse)
adata_pt = adata[keep].copy()
adata_pt.obs['comparison_condition'] = np.where(
    adata_pt.obs['condition'].astype(str).str.lower().eq('control'), 'healthy', 'aki'
)
adata_pt.obs['shared_pseudospace'] = adata_pt.obs[CONFIG['pseudotime_col']].to_numpy(dtype=float)
if not adata_pt.obs['shared_pseudospace'].between(0, 1, inclusive='both').all():
    raise ValueError(f"{CONFIG['pseudotime_col']} must already be normalized to [0, 1]")

cohort_summary = (
    adata_pt.obs.groupby(['comparison_condition', 'condition', 'sample'], observed=True)
    .size().rename('n_pt_tubules').reset_index()
)
cohort_summary.to_csv(OUTPUT_DIR / 'cohort_summary.csv', index=False)
print(f'Loaded canonical filtered DPT object: {INPUT_PATH.relative_to(PROJECT_DIR)}')
print(f'Filtered matrix: {adata.n_obs:,} tubules x {adata.n_vars:,} ortholog genes')
print(f'Mouse healthy-vs-AKI PT comparison cohort: {adata_pt.n_obs:,} tubules')
print(cohort_summary.to_string(index=False))

## GAM estimation and statistical testing

Penalized Gaussian GAMs estimate smooth expression trajectories from individual tubules.
The smooth basis, common-support prediction grid, expression filters, and effect-size
summaries are unchanged.

- Dynamic-trajectory permutations shuffle pseudospace within each sample, preserving
  sample-specific expression and pseudospace distributions.
- Benjamini-Hochberg correction is applied to blocked permutation p-values.
- Genes and pathway scores are called statistically dynamic using permutation FDR.
- Statistically dynamic results are ranked by fitted amplitude or by the root-mean-square
  healthy-versus-AKI curve difference.
- Pathway overrepresentation FDR identifies statistically significant pathways, which are
  ranked by enrichment fold and fitted trajectory effect size.

With only two samples per condition, healthy-versus-AKI curve differences are reported as
descriptive effect sizes rather than inferential claims.


### GAM Helper Functions

Defines penalized GAM smooth, binning, dynamic-permutation, and curve-correlation helpers used by the gene-level and pathway-level experiments. The output confirms the helper setup.

### Matrix, GAM Smooth, and Binning Utilities

Defines sparse conversion, row z-scoring, penalized B-spline GAM design matrices, and pseudospace binning helpers.

In [ ]:
def as_csr(matrix):
    if sp.issparse(matrix):
        return matrix.tocsr().astype(np.float64)
    return sp.csr_matrix(np.asarray(matrix, dtype=np.float64))


def zscore_rows(values):
    values = np.asarray(values, dtype=float)
    mean = np.nanmean(values, axis=1, keepdims=True)
    std = np.nanstd(values, axis=1, keepdims=True)
    std[~np.isfinite(std) | (std == 0)] = 1.0
    return (values - mean) / std


def make_gam_design(x, knots):
    smooth = np.asarray(
        dmatrix(
            'bs(x, knots=knots, degree=3, include_intercept=False, '
            'lower_bound=0, upper_bound=1) - 1',
            {'x': np.asarray(x, dtype=float), 'knots': np.asarray(knots, dtype=float)},
        )
    )
    return np.column_stack([np.ones(len(x)), smooth])


def make_design(x, knots):
    return make_gam_design(x, knots)


def gam_internal_knots(x, basis_df=CONFIG['gam_basis_df']):
    degree = 3
    n_internal = max(int(basis_df) - degree, 0)
    if n_internal == 0:
        return np.array([], dtype=float)
    values = np.clip(np.asarray(x, dtype=float), 0, 1)
    probs = np.linspace(0, 1, n_internal + 2)[1:-1]
    knots = np.unique(np.clip(np.quantile(values, probs), 1e-6, 1 - 1e-6))
    if knots.size != n_internal:
        knots = np.linspace(0, 1, n_internal + 2)[1:-1]
    return knots


def second_difference_penalty(n_coefficients):
    if n_coefficients <= 2:
        return np.eye(n_coefficients)
    diff = np.diff(np.eye(n_coefficients), n=2, axis=0)
    return diff.T @ diff


def make_gam_penalty(n_columns, smooth_slices, n_observations, alpha=CONFIG['gam_alpha']):
    penalty = np.zeros((n_columns, n_columns), dtype=float)
    for smooth_slice in smooth_slices:
        start, stop, step = smooth_slice.indices(n_columns)
        if step != 1:
            raise ValueError('GAM smooth slices must be contiguous')
        n_smooth = stop - start
        if n_smooth <= 0:
            continue
        scale = float(alpha) * max(int(n_observations), 1) / max(n_smooth, 1)
        penalty[start:stop, start:stop] += scale * second_difference_penalty(n_smooth)
    return penalty


def single_smooth_slice(design):
    return [slice(1, design.shape[1])]


def make_bins(pseudospace, n_bins=CONFIG['n_bins']):
    values = np.asarray(pseudospace, dtype=float)
    ids = np.floor(np.clip(values, 0, 1 - np.finfo(float).eps) * n_bins).astype(int)
    centers = (np.arange(n_bins) + 0.5) / n_bins
    return ids, centers


def binned_means(matrix, pseudospace, n_bins=CONFIG['n_bins']):
    ids, centers = make_bins(pseudospace, n_bins=n_bins)
    one_hot = sp.csr_matrix(
        (np.ones(ids.size), (np.arange(ids.size), ids)), shape=(ids.size, n_bins)
    )
    counts = np.asarray(one_hot.sum(axis=0)).ravel()
    sums = matrix.T @ one_hot
    if sp.issparse(sums):
        sums = sums.toarray()
    means = np.asarray(sums) / np.maximum(counts, 1)[None, :]
    means[:, counts == 0] = np.nan
    return means, counts, centers
print('Matrix, GAM smooth, and binning utilities are defined.')

### Sparse GAM and Blocked-Permutation Utilities

Defines vectorized penalized GAM fitting and empirical dynamic tests that shuffle
pseudospace within biological samples. The default backend evaluates 10,000 permutations
on CUDA GPU 0 in float32 batches. Setting `CONFIG['permutation_backend'] = 'cpu'`
uses the cached threaded CPU implementation.


In [ ]:
def sparse_gam_sse(matrix, design, smooth_slices=None):
    # Fit all responses together through penalized sparse cross-products: matrix is tubules x genes.
    values = as_csr(matrix)
    x = np.asarray(design, dtype=float)
    smooth_slices = [] if smooth_slices is None else smooth_slices
    xtx = x.T @ x
    penalty = make_gam_penalty(x.shape[1], smooth_slices, x.shape[0])
    system = xtx + penalty
    system_inv = np.linalg.pinv(system)
    xty = np.asarray(x.T @ values)
    beta = np.asarray(system_inv @ xty)
    yty = np.asarray(values.multiply(values).sum(axis=0)).ravel()
    fitted_cross = np.sum(beta * xty, axis=0)
    model_cross = np.sum(beta * (xtx @ beta), axis=0)
    sse = yty - 2 * fitted_cross + model_cross
    edf = float(np.trace(system_inv @ xtx))
    return np.maximum(sse, 0.0), beta, edf


def sparse_gam_nested_f_test(matrix, reduced_design, full_design, reduced_smooth_slices=None, full_smooth_slices=None):
    sse_reduced, _, edf_reduced = sparse_gam_sse(matrix, reduced_design, reduced_smooth_slices)
    sse_full, beta_full, edf_full = sparse_gam_sse(matrix, full_design, full_smooth_slices)
    n = matrix.shape[0]
    df_num = max(edf_full - edf_reduced, 1e-6)
    df_den = max(n - edf_full, 1.0)
    numerator = np.maximum(sse_reduced - sse_full, 0) / df_num
    denominator = np.maximum(sse_full / df_den, 1e-12)
    f_stat = numerator / denominator
    return f_stat, beta_full, edf_full


def permute_within_blocks(values, blocks, rng):
    values = np.asarray(values)
    blocks = np.asarray(blocks).astype(str)
    shuffled = values.copy()
    for block in np.unique(blocks):
        idx = np.flatnonzero(blocks == block)
        shuffled[idx] = rng.permutation(values[idx])
    return shuffled


def _permutation_chunks(n_permutations, n_workers):
    n_workers = max(1, min(int(n_workers), int(n_permutations)))
    base, remainder = divmod(int(n_permutations), n_workers)
    return [base + (worker < remainder) for worker in range(n_workers)]


def _gpu_permutation_exceedances(
    values,
    x,
    xtx,
    system_inv,
    yty,
    sse_reduced,
    df_num,
    df_den,
    observed_f,
    block_indices,
    n_permutations,
    rng,
    gpu_ids,
    batch_size,
):
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA permutation backend requested, but PyTorch cannot access CUDA')
    available = torch.cuda.device_count()
    devices = [int(device) for device in gpu_ids if int(device) < available]
    if not devices:
        raise RuntimeError(f'No requested CUDA devices are available; PyTorch reports {available}')

    dense_values = (
        values.toarray()
        if sp.issparse(values)
        else np.asarray(values)
    ).astype(np.float32, copy=False)
    x32 = np.asarray(x, dtype=np.float32)
    xtx32 = np.asarray(xtx, dtype=np.float32)
    system_inv32 = np.asarray(system_inv, dtype=np.float32)
    yty32 = np.asarray(yty, dtype=np.float32)
    sse_reduced32 = np.asarray(sse_reduced, dtype=np.float32)
    observed_f32 = np.asarray(observed_f, dtype=np.float32)
    chunks = _permutation_chunks(n_permutations, len(devices))
    seeds = rng.integers(0, np.iinfo(np.uint32).max, size=len(devices), dtype=np.uint32)

    def run_device(device_id, seed, chunk_size):
        device = torch.device(f'cuda:{device_id}')
        local_rng = np.random.default_rng(seed)
        with torch.cuda.device(device), torch.inference_mode():
            values_gpu = torch.as_tensor(dense_values, device=device)
            xtx_gpu = torch.as_tensor(xtx32, device=device)
            system_inv_gpu = torch.as_tensor(system_inv32, device=device)
            yty_gpu = torch.as_tensor(yty32, device=device)
            sse_reduced_gpu = torch.as_tensor(sse_reduced32, device=device)
            observed_f_gpu = torch.as_tensor(observed_f32, device=device)
            exceedances = torch.zeros(values.shape[1], dtype=torch.int64, device=device)
            batch_size_actual = max(1, min(int(batch_size), int(chunk_size)))

            for batch_start in range(0, chunk_size, batch_size_actual):
                current_batch = min(batch_size_actual, chunk_size - batch_start)
                permutations = np.broadcast_to(
                    np.arange(values.shape[0], dtype=np.int64),
                    (current_batch, values.shape[0]),
                ).copy()
                for batch_i in range(current_batch):
                    for idx in block_indices:
                        permutations[batch_i, idx] = local_rng.permutation(idx)
                stacked_design_t = (
                    x32[permutations]
                    .transpose(0, 2, 1)
                    .reshape(current_batch * x32.shape[1], x32.shape[0])
                )
                design_gpu = torch.as_tensor(stacked_design_t, device=device)
                xty = (design_gpu @ values_gpu).reshape(
                    current_batch, x32.shape[1], values.shape[1]
                )
                beta = torch.einsum('ij,bjg->big', system_inv_gpu, xty)
                fitted_cross = torch.einsum('big,big->bg', beta, xty)
                model_cross = torch.einsum('big,ij,bjg->bg', beta, xtx_gpu, beta)
                sse_full = torch.clamp(
                    yty_gpu.unsqueeze(0) - 2 * fitted_cross + model_cross,
                    min=0,
                )
                numerator = torch.clamp(
                    sse_reduced_gpu.unsqueeze(0) - sse_full,
                    min=0,
                ) / df_num
                denominator = torch.clamp(sse_full / df_den, min=1e-12)
                exceedances += ((numerator / denominator) >= observed_f_gpu).sum(dim=0)

            result = exceedances.cpu().numpy()
            del values_gpu, exceedances
            torch.cuda.empty_cache()
            return result

    with ThreadPoolExecutor(max_workers=len(devices)) as executor:
        counts = executor.map(run_device, devices, map(int, seeds), chunks)
        return np.sum(list(counts), axis=0, dtype=np.int64)

def empirical_dynamic_pvalues(
    matrix,
    pseudospace,
    knots,
    observed_f,
    n_permutations,
    rng,
    sample_blocks,
    n_workers=1,
    batch_size=1,
    backend='cpu',
    gpu_ids=(0,),
    gpu_batch_size=64,
):
    values = as_csr(matrix)
    base_design = make_gam_design(pseudospace, knots)
    smooth_slices = single_smooth_slice(base_design)
    null = np.ones((pseudospace.size, 1))

    # These quantities are invariant to row permutations and are computed once.
    sse_reduced, _, edf_reduced = sparse_gam_sse(values, null, [])
    x = np.asarray(base_design, dtype=float)
    xtx = x.T @ x
    penalty = make_gam_penalty(x.shape[1], smooth_slices, x.shape[0])
    system_inv = np.linalg.pinv(xtx + penalty)
    edf_full = float(np.trace(system_inv @ xtx))
    yty = np.asarray(values.multiply(values).sum(axis=0)).ravel()
    df_num = max(edf_full - edf_reduced, 1e-6)
    df_den = max(values.shape[0] - edf_full, 1.0)
    block_indices = [
        np.flatnonzero(np.asarray(sample_blocks).astype(str) == block)
        for block in np.unique(np.asarray(sample_blocks).astype(str))
    ]

    if backend == 'cuda':
        exceedances = _gpu_permutation_exceedances(
            values,
            x,
            xtx,
            system_inv,
            yty,
            sse_reduced,
            df_num,
            df_den,
            observed_f,
            block_indices,
            n_permutations,
            rng,
            gpu_ids,
            gpu_batch_size,
        )
        return (exceedances + 1) / (n_permutations + 1)

    def run_chunk(seed, chunk_size):
        local_rng = np.random.default_rng(seed)
        exceedances = np.zeros(values.shape[1], dtype=np.int32)
        batch_size_actual = max(1, min(int(batch_size), int(chunk_size)))
        for batch_start in range(0, chunk_size, batch_size_actual):
            current_batch = min(batch_size_actual, chunk_size - batch_start)
            stacked_design_t = np.empty(
                (current_batch * x.shape[1], x.shape[0]),
                dtype=x.dtype,
            )
            for batch_i in range(current_batch):
                permutation = np.arange(values.shape[0])
                for idx in block_indices:
                    permutation[idx] = local_rng.permutation(idx)
                stacked_design_t[
                    batch_i * x.shape[1]:(batch_i + 1) * x.shape[1]
                ] = x[permutation].T

            xty = np.asarray(stacked_design_t @ values).reshape(
                current_batch, x.shape[1], values.shape[1]
            )
            beta = np.einsum('ij,bjg->big', system_inv, xty, optimize=True)
            fitted_cross = np.einsum('big,big->bg', beta, xty, optimize=True)
            model_cross = np.einsum(
                'big,ij,bjg->bg', beta, xtx, beta, optimize=True
            )
            sse_full = np.maximum(
                yty[None, :] - 2 * fitted_cross + model_cross,
                0.0,
            )
            numerator = np.maximum(sse_reduced[None, :] - sse_full, 0.0) / df_num
            denominator = np.maximum(sse_full / df_den, 1e-12)
            exceedances += np.sum(
                (numerator / denominator) >= observed_f[None, :],
                axis=0,
                dtype=np.int32,
            )
        return exceedances

    chunks = _permutation_chunks(n_permutations, n_workers)
    seeds = rng.integers(0, np.iinfo(np.uint32).max, size=len(chunks), dtype=np.uint32)
    if len(chunks) == 1:
        exceedances = run_chunk(int(seeds[0]), chunks[0])
    else:
        with ThreadPoolExecutor(max_workers=len(chunks)) as executor:
            counts = executor.map(run_chunk, map(int, seeds), chunks)
            exceedances = np.sum(list(counts), axis=0, dtype=np.int64)
    return (exceedances + 1) / (n_permutations + 1)
print('Cached parallel sparse GAM permutation utilities are defined.')


### Multiple-Testing and Correlation Utilities

Defines Benjamini-Hochberg adjustment and robust Spearman-correlation helpers.


In [ ]:
def bh_adjust(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    adjusted = np.ones_like(pvalues)
    valid = np.isfinite(pvalues)
    adjusted[valid] = multipletests(pvalues[valid], method='fdr_bh')[1]
    return adjusted


def safe_spearman(left, right):
    left = np.asarray(left, dtype=float)
    right = np.asarray(right, dtype=float)
    valid = np.isfinite(left) & np.isfinite(right)
    if valid.sum() < 3 or np.ptp(left[valid]) == 0 or np.ptp(right[valid]) == 0:
        return np.nan
    return spearmanr(left[valid], right[valid]).correlation


def rowwise_curve_correlation(left, right):
    left = np.asarray(left, dtype=float)
    right = np.asarray(right, dtype=float)
    if left.shape != right.shape or left.ndim != 2:
        raise ValueError('Curve matrices must be two-dimensional with matching shapes')
    return np.asarray([
        safe_spearman(left_row, right_row)
        for left_row, right_row in zip(left, right)
    ])


def resolve_present(requested, universe):
    lookup = {str(gene).upper(): gene for gene in universe}
    return [lookup[str(gene).upper()] for gene in requested if str(gene).upper() in lookup]
print('Multiple-testing and correlation utilities are defined.')

### Condition-Specific Gene GAM Fitter

Defines the gene-level condition-specific dynamic GAM fitting workflow.


In [ ]:
def fit_condition_curves(adata_obj):
    matrix = as_csr(adata_obj.layers['lognorm'])
    detected = np.asarray((matrix > 0).sum(axis=0)).ravel()
    means = np.asarray(matrix.mean(axis=0)).ravel()
    min_detected = int(np.ceil(CONFIG['min_detected_fraction'] * adata_obj.n_obs))
    test_mask = (detected >= min_detected) & (means >= CONFIG['min_mean_expression'])
    genes = adata_obj.var_names.to_numpy()[test_mask]
    matrix = matrix[:, test_mask]
    pooled_pseudospace = adata_obj.obs['shared_pseudospace'].to_numpy(dtype=float)
    gam_knots = gam_internal_knots(pooled_pseudospace)
    condition_support = [
        adata_obj.obs.loc[adata_obj.obs['comparison_condition'].eq(condition), 'shared_pseudospace']
        .to_numpy(dtype=float)
        for condition in ['healthy', 'aki']
    ]
    overlap_min = max(np.min(values) for values in condition_support)
    overlap_max = min(np.max(values) for values in condition_support)
    grid = np.linspace(overlap_min, overlap_max, CONFIG['grid_points'])
    grid_design = make_gam_design(grid, gam_knots)
    condition_results = {}
    rng = np.random.default_rng(CONFIG['permutation_seed'])
    for condition in ['healthy', 'aki']:
        subset = adata_obj.obs['comparison_condition'].eq(condition).to_numpy()
        pseudospace = adata_obj.obs.loc[subset, 'shared_pseudospace'].to_numpy(dtype=float)
        condition_matrix = matrix[subset]
        design = make_gam_design(pseudospace, gam_knots)
        null = np.ones((pseudospace.size, 1))
        f_stat, beta, edf = sparse_gam_nested_f_test(
            condition_matrix, null, design, [], single_smooth_slice(design)
        )
        permutation_pvalue = empirical_dynamic_pvalues(
            condition_matrix,
            pseudospace,
            gam_knots,
            f_stat,
            CONFIG['dynamic_permutations'],
            rng,
            adata_obj.obs.loc[subset, 'sample'].astype(str).to_numpy(),
            n_workers=CONFIG['permutation_workers'],
            batch_size=CONFIG['permutation_batch_size'],
            backend=CONFIG['permutation_backend'],
            gpu_ids=CONFIG['permutation_gpu_ids'],
            gpu_batch_size=CONFIG['permutation_gpu_batch_size'],
        )
        curves = (grid_design @ beta).T
        amplitude = np.ptp(curves, axis=1)
        permutation_padj = bh_adjust(permutation_pvalue)
        dynamic = permutation_padj < CONFIG['dynamic_fdr']
        means_by_bin, counts, centers = binned_means(condition_matrix, pseudospace)
        condition_results[condition] = {
            'binned_means': means_by_bin,
            'counts': counts,
            'centers': centers,
            'curves': curves,
            'f_stat': f_stat,
            'gam_edf': edf,
            'permutation_pvalue': permutation_pvalue,
            'permutation_padj': permutation_padj,
            'amplitude': amplitude,
            'dynamic': dynamic,
        }
    return genes, matrix, grid, condition_results
print('Condition-specific gene GAM fitter is defined.')

## Statistically dynamic trajectories and effect-size comparison

Within each condition, a gene is called statistically dynamic when its blocked permutation
FDR is below `0.05`. Every permutation shuffles
pseudospace separately within each specimen. Dynamic detection in one condition and
non-detection in the other is reported only as detection status; it is not evidence that
the trajectories differ between conditions.

The output also records `condition_effect_rms` and `condition_effect_max_abs`, which
summarize the fitted healthy-versus-AKI curve difference. These effect sizes, curve
correlations, amplitudes, and peak shifts rank and describe dynamic genes.

The full table is written to `ortholog_trajectory_classification.csv`. No condition
interaction hit table is produced because the available biological replication is
insufficient for a useful healthy-versus-AKI significance test.


In [ ]:
genes, test_matrix, grid, condition_results = fit_condition_curves(adata_pt)
healthy = condition_results['healthy']
aki = condition_results['aki']

curve_corr = rowwise_curve_correlation(healthy['curves'], aki['curves'])
healthy_peak = grid[np.argmax(healthy['curves'], axis=1)]
aki_peak = grid[np.argmax(aki['curves'], axis=1)]
peak_shift = aki_peak - healthy_peak
amplitude_diff = aki['amplitude'] - healthy['amplitude']
curve_difference = aki['curves'] - healthy['curves']
condition_effect_rms = np.sqrt(np.mean(curve_difference**2, axis=1))
condition_effect_max_abs = np.max(np.abs(curve_difference), axis=1)
amplitude_ratio = np.maximum(healthy['amplitude'], aki['amplitude']) / np.maximum(
    np.minimum(healthy['amplitude'], aki['amplitude']), 1e-6
)

ortholog_stats = pd.DataFrame({
    'gene': genes,
    'healthy_dynamic_blocked_permutation_pvalue': healthy['permutation_pvalue'],
    'healthy_dynamic_blocked_permutation_fdr': healthy['permutation_padj'],
    'healthy_gam_edf': healthy['gam_edf'],
    'aki_dynamic_blocked_permutation_pvalue': aki['permutation_pvalue'],
    'aki_dynamic_blocked_permutation_fdr': aki['permutation_padj'],
    'aki_gam_edf': aki['gam_edf'],
    'healthy_dynamic': healthy['dynamic'],
    'aki_dynamic': aki['dynamic'],
    'healthy_amplitude': healthy['amplitude'],
    'aki_amplitude': aki['amplitude'],
    'amplitude_difference_aki_minus_healthy': amplitude_diff,
    'condition_effect_rms': condition_effect_rms,
    'condition_effect_max_abs': condition_effect_max_abs,
    'amplitude_ratio': amplitude_ratio,
    'curve_spearman': curve_corr,
    'healthy_peak_pseudospace': healthy_peak,
    'aki_peak_pseudospace': aki_peak,
    'peak_shift_aki_minus_healthy': peak_shift,
})


def classify(row):
    healthy_dynamic = bool(row['healthy_dynamic'])
    aki_dynamic = bool(row['aki_dynamic'])
    if healthy_dynamic and not aki_dynamic:
        return 'dynamic detected only in healthy'
    if aki_dynamic and not healthy_dynamic:
        return 'dynamic detected only in AKI'
    if not healthy_dynamic and not aki_dynamic:
        return 'dynamic not detected in either condition'
    if not np.isfinite(row['curve_spearman']):
        return 'undefined curve correlation'
    if row['curve_spearman'] < 0:
        return 'dynamic detected in both; opposite fitted directions'
    if abs(row['peak_shift_aki_minus_healthy']) >= CONFIG['peak_shift_threshold']:
        return 'dynamic detected in both; descriptive peak shift'
    if row['amplitude_ratio'] >= CONFIG['amplitude_ratio_threshold']:
        return 'dynamic detected in both; descriptive amplitude difference'
    if row['curve_spearman'] >= CONFIG['similar_curve_correlation']:
        return 'dynamic detected in both; similar fitted shape'
    return 'dynamic detected in both; dissimilar fitted shape'


ortholog_stats['trajectory_summary_class'] = ortholog_stats.apply(classify, axis=1)
ortholog_stats['statistically_dynamic'] = ortholog_stats['healthy_dynamic'] | ortholog_stats['aki_dynamic']
ortholog_stats = ortholog_stats.sort_values(
    ['statistically_dynamic', 'condition_effect_rms', 'curve_spearman'],
    ascending=[False, False, False],
)
ortholog_stats.to_csv(OUTPUT_DIR / 'ortholog_trajectory_classification.csv', index=False)
legacy_interaction_hits = OUTPUT_DIR / 'condition_pattern_interaction_hits.csv'
if legacy_interaction_hits.exists():
    legacy_interaction_hits.unlink()

classification_summary = (
    ortholog_stats['trajectory_summary_class'].value_counts().rename_axis('trajectory_summary_class')
    .rename('n_orthologs').reset_index()
)
classification_summary.to_csv(OUTPUT_DIR / 'ortholog_classification_summary.csv', index=False)

print(f'Tested {len(genes):,} expressed one-to-one orthologs')
print(f"Dynamic in healthy: {int(healthy['dynamic'].sum()):,}")
print(f"Dynamic in AKI: {int(aki['dynamic'].sum()):,}")
print(f"Dynamic detected in both conditions: {int((healthy['dynamic'] & aki['dynamic']).sum()):,}")
print('\nOrtholog trajectory classes:')
print(classification_summary.to_string(index=False))


## Top gene trajectory analysis and visualization

Top genes must be statistically dynamic by blocked permutation FDR in at least one
condition. They are then ranked by the root-mean-square fitted healthy-versus-AKI curve
difference.

The resulting plots are descriptive prioritization tools. They show group-average and
per-sample fitted curves but do not convert tubule count into biological replication.


### Select and export top genes

Selects the strongest group-pattern genes and exports their trajectory statistics and fitted standardized and raw curves.


In [ ]:
# Select statistically dynamic genes and rank them by fitted curve-difference effect size.
top_gene_columns = [
    'gene', 'trajectory_summary_class', 'condition_effect_rms',
    'condition_effect_max_abs', 'curve_spearman',
    'healthy_dynamic', 'aki_dynamic', 'healthy_amplitude', 'aki_amplitude',
    'healthy_peak_pseudospace', 'aki_peak_pseudospace',
    'peak_shift_aki_minus_healthy',
]
top_genes = ortholog_stats.loc[
    ortholog_stats['statistically_dynamic'], top_gene_columns
].head(CONFIG['gene_top_plot']).copy()
top_genes = top_genes.reset_index(drop=True)
top_genes.to_csv(OUTPUT_DIR / 'top_gene_trajectory_statistics.csv', index=False)

gene_index = {gene: i for i, gene in enumerate(genes)}
top_gene_indices = [gene_index[gene] for gene in top_genes['gene']]
top_gene_expression = test_matrix[:, top_gene_indices].tocsr().astype(np.float64)
top_gene_means = np.asarray(top_gene_expression.mean(axis=0)).ravel()
top_gene_sq_means = np.asarray(top_gene_expression.multiply(top_gene_expression).mean(axis=0)).ravel()
top_gene_stds = np.sqrt(np.maximum(top_gene_sq_means - top_gene_means**2, 0))
top_gene_stds[top_gene_stds < 1e-8] = 1.0

top_gene_curve_rows = []
for condition, result in condition_results.items():
    raw_curves = result['curves'][top_gene_indices]
    standardized_curves = (
        raw_curves - top_gene_means[:, None]
    ) / top_gene_stds[:, None]
    for gene_i, gene in enumerate(top_genes['gene']):
        for x, raw_value, standardized_value in zip(
            grid, raw_curves[gene_i], standardized_curves[gene_i]
        ):
            top_gene_curve_rows.append({
                'gene': gene,
                'condition': condition,
                'pseudospace': x,
                'fitted_gene_z_score': standardized_value,
                'fitted_raw_log_normalized_expression': raw_value,
            })
top_gene_curves = pd.DataFrame(top_gene_curve_rows)
top_gene_curves.to_csv(OUTPUT_DIR / 'top_gene_fitted_curves.csv', index=False)
display(top_genes)
display(top_gene_curves.head())


### Top gene trajectories

Plots the selected genes using pooled-cohort gene z-scores and raw log-normalized expression, matching the two pathway trajectory views.


In [ ]:
# Pathway-style group-level plots for the top effect-size-ranked dynamic genes.
ncols = 3
nrows = max(1, int(np.ceil(len(top_genes) / ncols)))

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.4 * nrows), sharex=True)
axes = np.atleast_1d(axes).ravel()
for ax, row in zip(axes, top_genes.itertuples(index=False)):
    subset = top_gene_curves.loc[top_gene_curves['gene'].eq(row.gene)]
    sns.lineplot(
        data=subset, x='pseudospace', y='fitted_gene_z_score',
        hue='condition', ax=ax, linewidth=2,
    )
    ax.set_title(f'{row.gene}: {row.trajectory_summary_class}', fontsize=9)
    ax.set_xlabel('Shared pseudospace')
    ax.set_ylabel('Gene z-score')
for ax in axes[len(top_genes):]:
    ax.axis('off')
fig.suptitle('Top effect-size-ranked dynamic gene trajectories', y=1.01)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'top_gene_zscore_trajectories.png', bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(OUTPUT_DIR / 'top_gene_zscore_trajectories.png')))

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.4 * nrows), sharex=True)
axes = np.atleast_1d(axes).ravel()
for ax, row in zip(axes, top_genes.itertuples(index=False)):
    subset = top_gene_curves.loc[top_gene_curves['gene'].eq(row.gene)]
    sns.lineplot(
        data=subset, x='pseudospace', y='fitted_raw_log_normalized_expression',
        hue='condition', ax=ax, linewidth=2,
    )
    ax.set_title(f'{row.gene}: {row.trajectory_summary_class}', fontsize=9)
    ax.set_xlabel('Shared pseudospace')
    ax.set_ylabel('Raw log-normalized expression')
for ax in axes[len(top_genes):]:
    ax.axis('off')
fig.suptitle('Top effect-size-ranked dynamic genes, raw log-normalized expression', y=1.01)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'top_gene_raw_lognorm_trajectories.png', bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(OUTPUT_DIR / 'top_gene_raw_lognorm_trajectories.png')))


### Per-sample top gene curves

Fits and plots sample-level sensitivity curves for the same selected genes.


In [ ]:
# Fit and plot sample-level sensitivity curves for the same top genes.
pooled_pseudospace = adata_pt.obs['shared_pseudospace'].to_numpy(dtype=float)
top_gene_gam_knots = gam_internal_knots(pooled_pseudospace)
top_gene_grid_design = make_gam_design(grid, top_gene_gam_knots)
sample_top_gene_rows = []
for sample in adata_pt.obs['sample'].astype(str).unique():
    sample_mask = adata_pt.obs['sample'].astype(str).eq(sample).to_numpy()
    sample_pseudospace = pooled_pseudospace[sample_mask]
    sample_design = make_gam_design(sample_pseudospace, top_gene_gam_knots)
    _, sample_beta, _ = sparse_gam_sse(
        test_matrix[sample_mask][:, top_gene_indices],
        sample_design,
        single_smooth_slice(sample_design),
    )
    sample_raw_curves = (top_gene_grid_design @ sample_beta).T
    sample_standardized_curves = (
        sample_raw_curves - top_gene_means[:, None]
    ) / top_gene_stds[:, None]
    supported = (grid >= np.min(sample_pseudospace)) & (grid <= np.max(sample_pseudospace))
    sample_standardized_curves[:, ~supported] = np.nan
    sample_condition = adata_pt.obs.loc[
        sample_mask, 'comparison_condition'
    ].astype(str).iloc[0]
    for gene_i, gene in enumerate(top_genes['gene']):
        for x, value in zip(grid, sample_standardized_curves[gene_i]):
            sample_top_gene_rows.append({
                'gene': gene,
                'sample': sample,
                'condition': sample_condition,
                'pseudospace': x,
                'fitted_gene_z_score': value,
            })
top_gene_per_sample_curves = pd.DataFrame(sample_top_gene_rows)
top_gene_per_sample_curves.to_csv(
    OUTPUT_DIR / 'top_gene_per_sample_curves.csv', index=False
)

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.4 * nrows), sharex=True)
axes = np.atleast_1d(axes).ravel()
for ax, row in zip(axes, top_genes.itertuples(index=False)):
    subset = top_gene_per_sample_curves.loc[
        top_gene_per_sample_curves['gene'].eq(row.gene)
    ]
    sns.lineplot(
        data=subset, x='pseudospace', y='fitted_gene_z_score',
        hue='sample', style='condition', ax=ax, linewidth=2,
    )
    ax.set_title(f'{row.gene}: {row.trajectory_summary_class}', fontsize=9)
    ax.set_xlabel('Shared pseudospace')
    ax.set_ylabel('Gene z-score')
for ax in axes[len(top_genes):]:
    ax.axis('off')
fig.suptitle('Per-sample curves for top effect-size-ranked dynamic genes', y=1.01)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'top_gene_per_sample_curves.png', bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(OUTPUT_DIR / 'top_gene_per_sample_curves.png')))

print(f'Top genes plotted: {len(top_genes):,}')
print(f"Statistically dynamic genes available: {int(ortholog_stats['statistically_dynamic'].sum()):,}")


## Canonical marker-gradient validation

The shared pseudospace coordinate is first validated using canonical proximal-tubule marker
genes. S1 is represented by `Slc5a2`; S2 by `Slc22a6`, `Slc34a1`, and `Gatm`; and S3 by
`Slc7a13` and `Acsm3`. For each available marker, the previously fitted condition-specific
GAM trajectory is exported and plotted across shared pseudospace. Marker genes that
are absent after filtering are omitted rather than restored from an unfiltered matrix.

Per-sample sensitivity plots are generated for the retained canonical markers. Each
available sample is modeled separately at tubule resolution using the same pooled GAM
knots and the same shared-support prediction grid used for the condition-level comparison.
Each displayed sample curve is masked outside that sample's observed pseudospace support
to avoid interpreting extrapolated tails.

### Canonical Marker Definitions

Declares the canonical proximal-tubule marker groups used for pseudospace validation plots.

In [ ]:
MARKER_GROUPS = {
    'S1': ['Slc5a2'],
    'S2': ['Slc22a6', 'Slc34a1', 'Gatm'],
    'S3': ['Slc7a13', 'Acsm3'],
}

gene_index = {gene: i for i, gene in enumerate(genes)}
print('Canonical marker groups are defined.')

### Canonical Marker Trajectory Table

Extracts condition-specific fitted GAM curves for available canonical marker genes.


In [ ]:
marker_rows = []
for group, requested in MARKER_GROUPS.items():
    for gene in resolve_present(requested, genes):
        idx = gene_index[gene]
        for condition, result in condition_results.items():
            for x, value in zip(grid, result['curves'][idx]):
                marker_rows.append({
                    'marker_group': group, 'gene': gene, 'condition': condition,
                    'pseudospace': x, 'fitted_expression': value,
                })
marker_curves = pd.DataFrame(marker_rows)
marker_curves.to_csv(OUTPUT_DIR / 'canonical_marker_curves.csv', index=False)
display(marker_curves.head())

### Per-Sample Marker Sensitivity Curves

Fits per-sample canonical marker curves to assess whether marker trajectories are driven by a single sample.

In [ ]:
# Fit sample-specific curves for every retained canonical marker gene.
sample_curve_genes = resolve_present(
    sorted({gene for requested in MARKER_GROUPS.values() for gene in requested}),
    genes,
)
sample_curve_indices = [gene_index[gene] for gene in sample_curve_genes]
sample_curve_index = {gene: i for i, gene in enumerate(sample_curve_genes)}
sample_gam_knots = gam_internal_knots(adata_pt.obs['shared_pseudospace'].to_numpy(dtype=float))
sample_grid_design = make_gam_design(grid, sample_gam_knots)
sample_gene_curves = {}
sample_condition = {}
for sample in adata_pt.obs['sample'].astype(str).unique():
    sample_mask = adata_pt.obs['sample'].astype(str).eq(sample).to_numpy()
    sample_pseudospace = adata_pt.obs.loc[sample_mask, 'shared_pseudospace'].to_numpy(dtype=float)
    sample_design = make_gam_design(sample_pseudospace, sample_gam_knots)
    _, sample_beta, _ = sparse_gam_sse(
        test_matrix[sample_mask][:, sample_curve_indices],
        sample_design,
        single_smooth_slice(sample_design),
    )
    fitted_sample_curves = (sample_grid_design @ sample_beta).T
    supported = (grid >= np.min(sample_pseudospace)) & (grid <= np.max(sample_pseudospace))
    fitted_sample_curves[:, ~supported] = np.nan
    sample_gene_curves[sample] = fitted_sample_curves
    sample_condition[sample] = adata_pt.obs.loc[sample_mask, 'comparison_condition'].astype(str).iloc[0]

sample_marker_rows = []
for group, requested in MARKER_GROUPS.items():
    for gene in resolve_present(requested, sample_curve_genes):
        gene_i = sample_curve_index[gene]
        for sample, curves in sample_gene_curves.items():
            for x, value in zip(grid, curves[gene_i]):
                sample_marker_rows.append({
                    'marker_group': group, 'gene': gene, 'sample': sample,
                    'condition': sample_condition[sample], 'pseudospace': x,
                    'fitted_expression': value,
                })
sample_marker_curves = pd.DataFrame(sample_marker_rows)
sample_marker_curves.to_csv(OUTPUT_DIR / 'per_sample_marker_curves.csv', index=False)
display(sample_marker_curves.head())

## Predefined pathway trajectory analysis

The modeled panel contains every eligible Hallmark, KEGG, and Reactome term with `10` to
`150` genes in the tested ortholog background. GO Biological Process remains available
for enrichment interpretation but is excluded from the modeled panel because of its
large, highly redundant term collection.

Gene-list overrepresentation FDR identifies statistically significant pathways.
Enrichment fold ranks significant terms. Pathway-score trajectories are tested separately
with pseudospace permutations blocked by sample; statistically dynamic pathway scores are
ranked by fitted healthy-versus-AKI curve effect size.

Pathway scores are arithmetic means of member-gene z-scores. The source libraries do not
provide reliable activating and inhibitory signs, so these scores represent average
member expression rather than definitive pathway activity.

Cross-condition pathway differences are descriptive effect sizes because there are only
two biological samples per condition.


### Pathway Library Inputs

Loads cached GO/Reactome/Hallmark/KEGG libraries and declares curated PT gene sets.


In [ ]:
from scipy.stats import hypergeom

PATHWAY_LIBRARY_DIR = PROJECT_DIR / 'data' / 'mouse_vs_human' / 'pathway_gene_sets'
PATHWAY_LIBRARY_FILES = {
    'GO_Biological_Process_2023': PATHWAY_LIBRARY_DIR / 'GO_Biological_Process_2023.json',
    'Reactome_2022': PATHWAY_LIBRARY_DIR / 'Reactome_2022.json',
    'MSigDB_Hallmark_2020': PATHWAY_LIBRARY_DIR / 'MSigDB_Hallmark_2020.json',
    'KEGG_2019_Mouse': PATHWAY_LIBRARY_DIR / 'KEGG_2019_Mouse.json',
}
missing_pathway_files = [str(path) for path in PATHWAY_LIBRARY_FILES.values() if not path.exists()]
if missing_pathway_files:
    raise FileNotFoundError(f'Missing cached pathway libraries: {missing_pathway_files}')

display(pd.DataFrame({'library': list(PATHWAY_LIBRARY_FILES.keys())}))

### Enrichment and Pathway-Model Helpers

Defines dense pathway-score model helpers, enrichment helpers, and pathway classification logic.


### Dense Pathway-Score GAM Utilities

Defines dense Gaussian GAM fitting, nested F-tests with effective degrees of freedom, and empirical dynamic p-values for pathway-score matrices.

In [ ]:
def dense_gam_sse(values, design, smooth_slices=None):
    values = np.asarray(values, dtype=np.float64)
    x = np.asarray(design, dtype=np.float64)
    smooth_slices = [] if smooth_slices is None else smooth_slices
    xtx = x.T @ x
    penalty = make_gam_penalty(x.shape[1], smooth_slices, x.shape[0])
    system = xtx + penalty
    system_inv = np.linalg.pinv(system)
    xty = x.T @ values
    beta = system_inv @ xty
    yty = np.sum(values * values, axis=0)
    fitted_cross = np.sum(beta * xty, axis=0)
    model_cross = np.sum(beta * (xtx @ beta), axis=0)
    sse = yty - 2 * fitted_cross + model_cross
    edf = float(np.trace(system_inv @ xtx))
    return np.maximum(sse, 0.0), beta, edf


def dense_gam_nested_f_test(values, reduced_design, full_design, reduced_smooth_slices=None, full_smooth_slices=None):
    sse_reduced, _, edf_reduced = dense_gam_sse(values, reduced_design, reduced_smooth_slices)
    sse_full, beta_full, edf_full = dense_gam_sse(values, full_design, full_smooth_slices)
    df_num = max(edf_full - edf_reduced, 1e-6)
    df_den = max(values.shape[0] - edf_full, 1.0)
    numerator = np.maximum(sse_reduced - sse_full, 0) / df_num
    denominator = np.maximum(sse_full / df_den, 1e-12)
    f_stat = numerator / denominator
    return f_stat, beta_full, edf_full


def dense_empirical_dynamic_pvalues(
    values,
    pseudospace,
    knots,
    observed_f,
    n_permutations,
    rng,
    sample_blocks,
    n_workers=1,
    batch_size=1,
    backend='cpu',
    gpu_ids=(0,),
    gpu_batch_size=64,
):
    values = np.asarray(values, dtype=np.float64)
    base_design = make_gam_design(pseudospace, knots)
    smooth_slices = single_smooth_slice(base_design)
    null = np.ones((pseudospace.size, 1))

    sse_reduced, _, edf_reduced = dense_gam_sse(values, null, [])
    x = np.asarray(base_design, dtype=np.float64)
    xtx = x.T @ x
    penalty = make_gam_penalty(x.shape[1], smooth_slices, x.shape[0])
    system_inv = np.linalg.pinv(xtx + penalty)
    edf_full = float(np.trace(system_inv @ xtx))
    yty = np.sum(values * values, axis=0)
    df_num = max(edf_full - edf_reduced, 1e-6)
    df_den = max(values.shape[0] - edf_full, 1.0)
    block_indices = [
        np.flatnonzero(np.asarray(sample_blocks).astype(str) == block)
        for block in np.unique(np.asarray(sample_blocks).astype(str))
    ]

    if backend == 'cuda':
        exceedances = _gpu_permutation_exceedances(
            values,
            x,
            xtx,
            system_inv,
            yty,
            sse_reduced,
            df_num,
            df_den,
            observed_f,
            block_indices,
            n_permutations,
            rng,
            gpu_ids,
            gpu_batch_size,
        )
        return (exceedances + 1) / (n_permutations + 1)

    def run_chunk(seed, chunk_size):
        local_rng = np.random.default_rng(seed)
        exceedances = np.zeros(values.shape[1], dtype=np.int32)
        batch_size_actual = max(1, min(int(batch_size), int(chunk_size)))
        for batch_start in range(0, chunk_size, batch_size_actual):
            current_batch = min(batch_size_actual, chunk_size - batch_start)
            stacked_design_t = np.empty(
                (current_batch * x.shape[1], x.shape[0]),
                dtype=x.dtype,
            )
            for batch_i in range(current_batch):
                permutation = np.arange(values.shape[0])
                for idx in block_indices:
                    permutation[idx] = local_rng.permutation(idx)
                stacked_design_t[
                    batch_i * x.shape[1]:(batch_i + 1) * x.shape[1]
                ] = x[permutation].T

            xty = np.asarray(stacked_design_t @ values).reshape(
                current_batch, x.shape[1], values.shape[1]
            )
            beta = np.einsum('ij,bjg->big', system_inv, xty, optimize=True)
            fitted_cross = np.einsum('big,big->bg', beta, xty, optimize=True)
            model_cross = np.einsum(
                'big,ij,bjg->bg', beta, xtx, beta, optimize=True
            )
            sse_full = np.maximum(
                yty[None, :] - 2 * fitted_cross + model_cross,
                0.0,
            )
            numerator = np.maximum(sse_reduced[None, :] - sse_full, 0.0) / df_num
            denominator = np.maximum(sse_full / df_den, 1e-12)
            exceedances += np.sum(
                (numerator / denominator) >= observed_f[None, :],
                axis=0,
                dtype=np.int32,
            )
        return exceedances

    chunks = _permutation_chunks(n_permutations, n_workers)
    seeds = rng.integers(0, np.iinfo(np.uint32).max, size=len(chunks), dtype=np.uint32)
    if len(chunks) == 1:
        exceedances = run_chunk(int(seeds[0]), chunks[0])
    else:
        with ThreadPoolExecutor(max_workers=len(chunks)) as executor:
            counts = executor.map(run_chunk, map(int, seeds), chunks)
            exceedances = np.sum(list(counts), axis=0, dtype=np.int64)
    return (exceedances + 1) / (n_permutations + 1)

print('Cached parallel dense pathway-score permutation utilities are defined.')


### Pathway Classification and Gene-List Helpers

Defines descriptive pathway trajectory summaries and gene-list construction from within-group dynamic tests.


In [ ]:
def classify_pathway(row):
    if row['healthy_dynamic'] and not row['aki_dynamic']:
        return 'dynamic detected only in healthy'
    if row['aki_dynamic'] and not row['healthy_dynamic']:
        return 'dynamic detected only in AKI'
    if not row['healthy_dynamic'] and not row['aki_dynamic']:
        return 'dynamic not detected in either condition'
    if not np.isfinite(row['curve_spearman']):
        return 'undefined curve correlation'
    if row['curve_spearman'] < 0:
        return 'dynamic detected in both; opposite fitted directions'
    if abs(row['peak_shift_aki_minus_healthy']) >= CONFIG['peak_shift_threshold']:
        return 'dynamic detected in both; descriptive peak shift'
    if row['amplitude_ratio'] >= CONFIG['amplitude_ratio_threshold']:
        return 'dynamic detected in both; descriptive amplitude difference'
    if row['curve_spearman'] >= CONFIG['similar_curve_correlation']:
        return 'dynamic detected in both; similar fitted shape'
    return 'dynamic detected in both; dissimilar fitted shape'


def build_dynamic_gene_lists(stats):
    both_dynamic = stats['healthy_dynamic'] & stats['aki_dynamic']
    return {
        'healthy_dynamic_genes': stats.loc[stats['healthy_dynamic'], 'gene'].tolist(),
        'aki_dynamic_genes': stats.loc[stats['aki_dynamic'], 'gene'].tolist(),
        'dynamic_both_similar_shape_genes': stats.loc[
            both_dynamic & (stats['curve_spearman'] >= CONFIG['similar_curve_correlation']), 'gene'
        ].tolist(),
        'dynamic_detected_only_healthy': stats.loc[
            stats['healthy_dynamic'] & ~stats['aki_dynamic'], 'gene'
        ].tolist(),
        'dynamic_detected_only_aki': stats.loc[
            stats['aki_dynamic'] & ~stats['healthy_dynamic'], 'gene'
        ].tolist(),
        'peak_shifted_genes': stats.loc[
            both_dynamic & (stats['peak_shift_aki_minus_healthy'].abs() >= CONFIG['peak_shift_threshold']), 'gene'
        ].tolist(),
    }
print('Pathway classification and gene-list helpers are defined.')

### Functional Enrichment Runner

Defines hypergeometric overrepresentation testing against the tested-ortholog background.


In [ ]:
def normalize_gene_set(requested_genes, background_lookup):
    return sorted({
        background_lookup[gene.upper()]
        for gene in requested_genes
        if isinstance(gene, str) and gene.upper() in background_lookup
    })


def run_functional_enrichment(raw_libraries, gene_lists, background_genes):
    background_lookup = {gene.upper(): gene for gene in background_genes}
    background_set = set(background_genes)
    background_size = len(background_set)
    rows = []
    eligible_terms = []
    for library, gene_sets in raw_libraries.items():
        for term, requested_genes in gene_sets.items():
            term_genes = normalize_gene_set(requested_genes, background_lookup)
            if CONFIG['pathway_min_genes'] <= len(term_genes) <= CONFIG['pathway_max_genes']:
                eligible_terms.append((library, term, term_genes, set(term_genes)))
    for gene_list_name, list_genes in gene_lists.items():
        query_genes = sorted(set(list_genes) & background_set)
        query_size = len(query_genes)
        query_set = set(query_genes)
        for library, term, term_genes, term_set in eligible_terms:
            overlap_genes = sorted(query_set & term_set)
            overlap = len(overlap_genes)
            if query_size == 0:
                pvalue = 1.0
            else:
                pvalue = hypergeom.sf(overlap - 1, background_size, len(term_genes), query_size)
            rows.append({
                'gene_list': gene_list_name,
                'library': library,
                'pathway': term,
                'background_size': background_size,
                'query_size': query_size,
                'term_size_background': len(term_genes),
                'overlap': overlap,
                'enrichment_overrepresentation_pvalue': pvalue,
                'enrichment_fold': (
                    (overlap / query_size) / (len(term_genes) / background_size)
                    if query_size and term_genes else np.nan
                ),
                'overlap_genes': ';'.join(overlap_genes),
                'genes_present': ';'.join(term_genes),
                'n_genes_present': len(term_genes),
                'gene_list_members': ';'.join(query_genes),
                'term_gene_list': term_genes,
            })
    enrichment = pd.DataFrame(rows)
    if enrichment.empty:
        return enrichment
    enrichment['enrichment_overrepresentation_fdr'] = np.nan
    for gene_list_name, idx in enrichment.groupby('gene_list', observed=True).groups.items():
        enrichment.loc[idx, 'enrichment_overrepresentation_fdr'] = bh_adjust(
            enrichment.loc[idx, 'enrichment_overrepresentation_pvalue'].to_numpy()
        )
    enrichment['significant_enrichment'] = (
        enrichment['enrichment_overrepresentation_fdr'] < CONFIG['enrichment_fdr']
    )
    return enrichment.sort_values(
        ['gene_list', 'significant_enrichment', 'enrichment_fold', 'enrichment_overrepresentation_fdr'],
        ascending=[True, False, False, True],
    )
print('Functional enrichment runner is defined.')

### Predefined Pathway Panel Builder

Builds the Hallmark, KEGG, and Reactome pathway panel independently of the observed gene
trajectory results. Enrichment metadata are attached only for interpretation.


In [ ]:
def build_predefined_pathway_panel(raw_libraries, background_genes, enrichment=None):
    background_lookup = {gene.upper(): gene for gene in background_genes}
    rows = []
    selected_libraries = set(CONFIG['pathway_model_libraries'])
    for library, gene_sets in raw_libraries.items():
        if library not in selected_libraries:
            continue
        for pathway, requested_genes in gene_sets.items():
            term_genes = normalize_gene_set(requested_genes, background_lookup)
            if CONFIG['pathway_min_genes'] <= len(term_genes) <= CONFIG['pathway_max_genes']:
                rows.append({
                    'library': library,
                    'pathway': pathway,
                    'genes_present': ';'.join(term_genes),
                    'n_genes_present': len(term_genes),
                    'gene_list': term_genes,
                })
    panel = pd.DataFrame(rows).sort_values(['library', 'pathway']).reset_index(drop=True)
    if panel.empty:
        raise ValueError('No predefined pathway terms passed the tested-gene size filters')

    panel['enriched_gene_lists'] = ''
    panel['best_enrichment_gene_list'] = pd.NA
    panel['best_enrichment_overrepresentation_pvalue'] = np.nan
    panel['best_enrichment_overrepresentation_fdr'] = np.nan
    panel['best_enrichment_overlap'] = 0
    panel['best_enrichment_fold'] = np.nan
    if enrichment is not None and not enrichment.empty:
        best = (
            enrichment.sort_values(
                ['significant_enrichment', 'enrichment_fold', 'enrichment_overrepresentation_fdr'],
                ascending=[False, False, True],
            )
            .drop_duplicates(['library', 'pathway'])
            .set_index(['library', 'pathway'])
        )
        for idx, row in panel.iterrows():
            key = (row['library'], row['pathway'])
            if key in best.index:
                match = best.loc[key]
                panel.at[idx, 'best_enrichment_gene_list'] = match['gene_list']
                panel.at[idx, 'best_enrichment_overrepresentation_pvalue'] = match[
                    'enrichment_overrepresentation_pvalue'
                ]
                panel.at[idx, 'best_enrichment_overrepresentation_fdr'] = match[
                    'enrichment_overrepresentation_fdr'
                ]
                panel.at[idx, 'best_enrichment_overlap'] = int(match['overlap'])
                panel.at[idx, 'best_enrichment_fold'] = match['enrichment_fold']
                matching = enrichment.loc[
                    enrichment['library'].eq(row['library'])
                    & enrichment['pathway'].eq(row['pathway'])
                    & enrichment['significant_enrichment'],
                    'gene_list',
                ].drop_duplicates()
                panel.at[idx, 'enriched_gene_lists'] = ';'.join(matching.astype(str))
    panel['significant_enrichment'] = (
        panel['best_enrichment_overrepresentation_fdr'] < CONFIG['enrichment_fdr']
    )
    return panel


print('Predefined pathway-panel builder is defined.')


### Run Enrichment and Define the Independent Pathway Panel

Exports gene-list enrichment results, then constructs the predefined pathway panel
without filtering on enrichment significance.


In [ ]:
raw_pathway_libraries = {
    library: json.loads(path.read_text())
    for library, path in PATHWAY_LIBRARY_FILES.items()
}

dynamic_gene_lists = build_dynamic_gene_lists(ortholog_stats)
gene_list_summary = pd.DataFrame([
    {'gene_list': name, 'n_genes': len(sorted(set(values) & set(genes)))}
    for name, values in dynamic_gene_lists.items()
])
gene_list_summary.to_csv(OUTPUT_DIR / 'functional_enrichment_gene_list_summary.csv', index=False)

# Enrichment remains an interpretation layer. It does not select pathways for modeling.
functional_enrichment = run_functional_enrichment(raw_pathway_libraries, dynamic_gene_lists, genes)
if functional_enrichment.empty:
    raise ValueError('No functional terms passed the background term-size thresholds')
functional_enrichment.drop(columns='term_gene_list').to_csv(
    OUTPUT_DIR / 'functional_enrichment_all_terms.csv', index=False
)
significant_enrichment = functional_enrichment.loc[
    functional_enrichment['significant_enrichment']
].copy()
significant_enrichment.drop(columns='term_gene_list').to_csv(
    OUTPUT_DIR / 'functional_enrichment_significant_terms.csv', index=False
)

retained_pathways = build_predefined_pathway_panel(
    raw_pathway_libraries,
    genes,
    enrichment=functional_enrichment,
)
retained_pathways.to_csv(
    OUTPUT_DIR / 'predefined_pathways_for_gam_modeling.csv',
    index=False,
)
legacy_selected_pathways = OUTPUT_DIR / 'prioritized_enriched_pathways_for_gam_modeling.csv'
if legacy_selected_pathways.exists():
    legacy_selected_pathways.unlink()

display(gene_list_summary)
display(retained_pathways.groupby('library', observed=True).size().rename('n_modeled_pathways'))
display(significant_enrichment[[
    'gene_list', 'library', 'pathway', 'overlap',
    'enrichment_overrepresentation_fdr', 'enrichment_fold',
]].head(20))


### Score Predefined Pathways

Converts every eligible predefined pathway into a tubule-level mean member-gene z-score.


In [ ]:
# Score tubules as the mean member-gene z-score without materializing a dense gene-z matrix.
pathway_gene_names = sorted({gene for values in retained_pathways['gene_list'] for gene in values})
pathway_gene_indices = [gene_index[gene] for gene in pathway_gene_names]
pathway_gene_local = {gene: i for i, gene in enumerate(pathway_gene_names)}
pathway_expression = test_matrix[:, pathway_gene_indices].tocsr().astype(np.float64)
gene_means = np.asarray(pathway_expression.mean(axis=0)).ravel()
gene_sq_means = np.asarray(pathway_expression.multiply(pathway_expression).mean(axis=0)).ravel()
gene_stds = np.sqrt(np.maximum(gene_sq_means - gene_means**2, 0))
gene_stds[gene_stds == 0] = 1.0
pathway_scores = np.empty((adata_pt.n_obs, len(retained_pathways)), dtype=np.float32)
for pathway_i, member_genes in enumerate(retained_pathways['gene_list']):
    member_indices = [pathway_gene_local[gene] for gene in member_genes]
    weights = 1.0 / gene_stds[member_indices]
    offset = np.mean(gene_means[member_indices] / gene_stds[member_indices])
    pathway_scores[:, pathway_i] = (
        np.asarray(pathway_expression[:, member_indices] @ weights).ravel() / len(member_indices) - offset
    )

display(retained_pathways[[
    'library', 'pathway', 'n_genes_present', 'best_enrichment_gene_list',
    'best_enrichment_overrepresentation_fdr', 'best_enrichment_fold',
    'significant_enrichment',
]].head(20))

### Run pathway GAM models

Fits group-specific pathway curves and blocked dynamic permutation tests. Permutation FDR
identifies statistically dynamic pathway scores; fitted curve differences rank them.


In [ ]:
if 'pathway_scores' not in globals():
    pathway_gene_names = sorted({gene for values in retained_pathways['gene_list'] for gene in values})
    pathway_gene_indices = [gene_index[gene] for gene in pathway_gene_names]
    pathway_gene_local = {gene: i for i, gene in enumerate(pathway_gene_names)}
    pathway_expression = test_matrix[:, pathway_gene_indices].tocsr().astype(np.float64)
    gene_means = np.asarray(pathway_expression.mean(axis=0)).ravel()
    gene_sq_means = np.asarray(pathway_expression.multiply(pathway_expression).mean(axis=0)).ravel()
    gene_stds = np.sqrt(np.maximum(gene_sq_means - gene_means**2, 0))
    gene_stds[gene_stds == 0] = 1.0
    pathway_scores = np.empty((adata_pt.n_obs, len(retained_pathways)), dtype=np.float32)
    for pathway_i, member_genes in enumerate(retained_pathways['gene_list']):
        member_indices = [pathway_gene_local[gene] for gene in member_genes]
        weights = 1.0 / gene_stds[member_indices]
        offset = np.mean(gene_means[member_indices] / gene_stds[member_indices])
        pathway_scores[:, pathway_i] = (
            np.asarray(pathway_expression[:, member_indices] @ weights).ravel() / len(member_indices) - offset
        )

pooled_pseudospace = adata_pt.obs['shared_pseudospace'].to_numpy(dtype=float)
pathway_gam_knots = gam_internal_knots(pooled_pseudospace)
pathway_grid_design = make_gam_design(grid, pathway_gam_knots)
pathway_condition_results = {}
pathway_rng = np.random.default_rng(CONFIG['permutation_seed'])
for condition in ['healthy', 'aki']:
    condition_mask = adata_pt.obs['comparison_condition'].eq(condition).to_numpy()
    condition_values = pathway_scores[condition_mask]
    condition_pseudospace = pooled_pseudospace[condition_mask]
    condition_design = make_gam_design(condition_pseudospace, pathway_gam_knots)
    null = np.ones((condition_pseudospace.size, 1))
    f_stat, beta, edf = dense_gam_nested_f_test(
        condition_values, null, condition_design, [], single_smooth_slice(condition_design)
    )
    permutation_pvalue = dense_empirical_dynamic_pvalues(
        condition_values,
        condition_pseudospace,
        pathway_gam_knots,
        f_stat,
        CONFIG['dynamic_permutations'],
        pathway_rng,
        adata_pt.obs.loc[condition_mask, 'sample'].astype(str).to_numpy(),
        n_workers=CONFIG['permutation_workers'],
        batch_size=CONFIG['permutation_batch_size'],
        backend=CONFIG['permutation_backend'],
        gpu_ids=CONFIG['permutation_gpu_ids'],
        gpu_batch_size=CONFIG['permutation_gpu_batch_size'],
    )
    curves = (pathway_grid_design @ beta).T
    amplitude = np.ptp(curves, axis=1)
    permutation_padj = bh_adjust(permutation_pvalue)
    pathway_condition_results[condition] = {
        'curves': curves,
        'amplitude': amplitude,
        'gam_edf': edf,
        'permutation_pvalue': permutation_pvalue,
        'permutation_padj': permutation_padj,
        'dynamic': permutation_padj < CONFIG['dynamic_fdr'],
    }

pathway_healthy = pathway_condition_results['healthy']
pathway_aki = pathway_condition_results['aki']
pathway_corr = rowwise_curve_correlation(pathway_healthy['curves'], pathway_aki['curves'])
pathway_healthy_peak = grid[np.argmax(pathway_healthy['curves'], axis=1)]
pathway_aki_peak = grid[np.argmax(pathway_aki['curves'], axis=1)]
pathway_amplitude_ratio = np.maximum(pathway_healthy['amplitude'], pathway_aki['amplitude']) / np.maximum(
    np.minimum(pathway_healthy['amplitude'], pathway_aki['amplitude']), 1e-6
)
pathway_curve_difference = pathway_aki['curves'] - pathway_healthy['curves']
pathway_condition_effect_rms = np.sqrt(np.mean(pathway_curve_difference**2, axis=1))
pathway_condition_effect_max_abs = np.max(np.abs(pathway_curve_difference), axis=1)
pathway_stats = retained_pathways.drop(columns='gene_list').copy()
pathway_stats = pathway_stats.assign(
    healthy_dynamic_blocked_permutation_pvalue=pathway_healthy['permutation_pvalue'],
    healthy_dynamic_blocked_permutation_fdr=pathway_healthy['permutation_padj'],
    healthy_gam_edf=pathway_healthy['gam_edf'],
    aki_dynamic_blocked_permutation_pvalue=pathway_aki['permutation_pvalue'],
    aki_dynamic_blocked_permutation_fdr=pathway_aki['permutation_padj'],
    aki_gam_edf=pathway_aki['gam_edf'],
    healthy_dynamic=pathway_healthy['dynamic'],
    aki_dynamic=pathway_aki['dynamic'],
    healthy_amplitude=pathway_healthy['amplitude'],
    aki_amplitude=pathway_aki['amplitude'],
    amplitude_difference_aki_minus_healthy=pathway_aki['amplitude'] - pathway_healthy['amplitude'],
    amplitude_ratio=pathway_amplitude_ratio,
    curve_spearman=pathway_corr,
    healthy_peak_pseudospace=pathway_healthy_peak,
    aki_peak_pseudospace=pathway_aki_peak,
    peak_shift_aki_minus_healthy=pathway_aki_peak - pathway_healthy_peak,
    condition_effect_rms=pathway_condition_effect_rms,
    condition_effect_max_abs=pathway_condition_effect_max_abs,
)
pathway_stats['trajectory_summary_class'] = pathway_stats.apply(classify_pathway, axis=1)
pathway_stats = pathway_stats.sort_values(
    ['significant_enrichment', 'condition_effect_rms', 'curve_spearman'],
    ascending=[False, False, False],
)
pathway_stats.to_csv(OUTPUT_DIR / 'pathway_trajectory_classification.csv', index=False)
legacy_pathway_hits = OUTPUT_DIR / 'pathway_condition_pattern_interaction_hits.csv'
if legacy_pathway_hits.exists():
    legacy_pathway_hits.unlink()

pathway_library_summary = (
    pathway_stats.groupby('library', observed=True)
    .agg(
        n_pathways=('pathway', 'size'),
        n_healthy_dynamic=('healthy_dynamic', 'sum'),
        n_aki_dynamic=('aki_dynamic', 'sum'),
        n_dynamic_both=('pathway', lambda values: int(
            (pathway_stats.loc[values.index, 'healthy_dynamic'] & pathway_stats.loc[values.index, 'aki_dynamic']).sum()
        )),
    )
    .reset_index()
)
pathway_library_summary.to_csv(OUTPUT_DIR / 'pathway_library_summary.csv', index=False)

pathway_display_columns = [
    'library', 'pathway', 'trajectory_summary_class',
    'best_enrichment_overrepresentation_fdr', 'best_enrichment_fold',
    'significant_enrichment', 'condition_effect_rms', 'curve_spearman',
    'healthy_dynamic', 'aki_dynamic',
]
display(pathway_stats[pathway_display_columns].head(20))
display(pathway_library_summary)

### Select top pathways

Selects pathways that pass enrichment FDR, then ranks them by fitted trajectory effect
size and exports their fitted curves.


In [ ]:
pathway_library_summary.to_csv(OUTPUT_DIR / 'pathway_library_summary.csv', index=False)

curve_rows = []
for pathway_i, row in retained_pathways.iterrows():
    for condition, result in pathway_condition_results.items():
        for x, value in zip(grid, result['curves'][pathway_i]):
            curve_rows.append({
                'library': row['library'], 'pathway': row['pathway'], 'condition': condition,
                'pseudospace': x, 'fitted_pathway_score': value,
            })
pathway_fitted_curves = pd.DataFrame(curve_rows)
pathway_fitted_curves.to_csv(OUTPUT_DIR / 'pathway_fitted_curves.csv', index=False)

top_pathways = pathway_stats.loc[pathway_stats['significant_enrichment']].head(
    CONFIG['pathway_top_plot']
)[['library', 'pathway']]
if top_pathways.empty:
    top_pathways = pathway_stats.loc[
        pathway_stats['healthy_dynamic'] | pathway_stats['aki_dynamic']
    ].head(CONFIG['pathway_top_plot'])[['library', 'pathway']]
top_pathway_curves = pathway_fitted_curves.merge(top_pathways, on=['library', 'pathway'], how='inner')
display(top_pathways)

### Build Per-Sample Pathway GAM Curves

Fits sample-level GAM sensitivity curves for the selected predefined pathways.

In [ ]:
# Sample-level sensitivity curves for the top effect-size-ranked pathways.
top_indices = [
    retained_pathways.index[
        (retained_pathways['library'].eq(row.library)) & (retained_pathways['pathway'].eq(row.pathway))
    ][0]
    for row in top_pathways.itertuples(index=False)
]
sample_pathway_rows = []
for sample in adata_pt.obs['sample'].astype(str).unique():
    sample_mask = adata_pt.obs['sample'].astype(str).eq(sample).to_numpy()
    sample_pseudospace = pooled_pseudospace[sample_mask]
    sample_design = make_gam_design(sample_pseudospace, pathway_gam_knots)
    _, sample_beta, _ = dense_gam_sse(
        pathway_scores[sample_mask][:, top_indices],
        sample_design,
        single_smooth_slice(sample_design),
    )
    sample_curves = (pathway_grid_design @ sample_beta).T
    supported = (grid >= np.min(sample_pseudospace)) & (grid <= np.max(sample_pseudospace))
    sample_curves[:, ~supported] = np.nan
    sample_condition_label = adata_pt.obs.loc[sample_mask, 'comparison_condition'].astype(str).iloc[0]
    for curve_i, row in enumerate(top_pathways.itertuples(index=False)):
        for x, value in zip(grid, sample_curves[curve_i]):
            sample_pathway_rows.append({
                'library': row.library, 'pathway': row.pathway, 'sample': sample,
                'condition': sample_condition_label, 'pseudospace': x, 'fitted_pathway_score': value,
            })
top_sample_pathway_curves = pd.DataFrame(sample_pathway_rows)
top_sample_pathway_curves.to_csv(OUTPUT_DIR / 'top_pathway_per_sample_curves.csv', index=False)
display(top_sample_pathway_curves.head())

### Pathway modeling summary

Reports statistically significant enrichment results and statistically dynamic pathway
scores from blocked permutation tests.


In [ ]:
print(f'Gene lists enriched: {len(dynamic_gene_lists):,}')
print(f'Significant enriched term instances: {len(significant_enrichment):,}')
print(f'Predefined pathway sets modeled: {len(pathway_stats):,}')
print(f"Dynamic pathways in healthy: {int(pathway_stats['healthy_dynamic'].sum()):,}")
print(f"Dynamic pathways in AKI: {int(pathway_stats['aki_dynamic'].sum()):,}")
print(f"Dynamic pathways in both: {int((pathway_stats['healthy_dynamic'] & pathway_stats['aki_dynamic']).sum()):,}")
print(f"Statistically significant enriched pathways: {int(pathway_stats['significant_enrichment'].sum()):,}")
print('\nPathway library summary:')
print(pathway_library_summary.to_string(index=False))

### Pathway Library Summary Figure

Plots one library-level summary figure for predefined pathway modeling.


In [ ]:
# Figure H: pathway-library summary.
summary_plot = pathway_library_summary.melt(
    id_vars='library',
    value_vars=['n_pathways', 'n_healthy_dynamic', 'n_aki_dynamic', 'n_dynamic_both'],
    var_name='metric', value_name='count',
)
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=summary_plot, x='library', y='count', hue='metric', ax=ax)
ax.set(title='H. Statistically dynamic pathway trajectory summary', xlabel='Gene-set collection', ylabel='Pathway count')
ax.tick_params(axis='x', rotation=20)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'H_pathway_library_summary.png', bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(OUTPUT_DIR / 'H_pathway_library_summary.png')))

### Top Predefined Pathway Trajectories

Plots selected predefined pathway terms first as mean gene z-score trajectories, then as mean raw log-normalized expression trajectories using the same retained member genes.


In [ ]:
# Figure I: condition curves for top effect-size-ranked pathways.
ncols = 3
nrows = max(1, int(np.ceil(len(top_pathways) / ncols)))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.4 * nrows), sharex=True)
axes = np.atleast_1d(axes).ravel()
for ax, row in zip(axes, top_pathways.itertuples(index=False)):
    subset = top_pathway_curves.loc[
        top_pathway_curves['library'].eq(row.library) & top_pathway_curves['pathway'].eq(row.pathway)
    ]
    sns.lineplot(data=subset, x='pseudospace', y='fitted_pathway_score', hue='condition', ax=ax, linewidth=2)
    ax.set_title(f'{row.library}: {row.pathway}', fontsize=9)
    ax.set_xlabel('Shared pseudospace')
    ax.set_ylabel('Mean gene z-score')
for ax in axes[len(top_pathways):]:
    ax.axis('off')
fig.suptitle('I. Top prioritized pathway trajectories', y=1.01)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'I_top_effect_size_pathway_trajectories.png', bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(OUTPUT_DIR / 'I_top_effect_size_pathway_trajectories.png')))

### Top Predefined Pathway Raw Log-Normalized Trajectories

Plots the same top predefined pathways as Figure I using the mean raw log-normalized expression of each pathway's retained member genes instead of the mean gene z-score.


In [ ]:
# Figure I2: raw log-normalized expression curves for top effect-size-ranked pathways.
top_indices = [
    retained_pathways.index[
        (retained_pathways['library'].eq(row.library)) & (retained_pathways['pathway'].eq(row.pathway))
    ][0]
    for row in top_pathways.itertuples(index=False)
]
raw_top_pathway_scores = np.empty((adata_pt.n_obs, len(top_indices)), dtype=np.float32)
for curve_i, pathway_i in enumerate(top_indices):
    member_genes = retained_pathways.loc[pathway_i, 'gene_list']
    member_indices = [gene_index[gene] for gene in member_genes]
    raw_top_pathway_scores[:, curve_i] = np.asarray(
        test_matrix[:, member_indices].mean(axis=1)
    ).ravel()

raw_curve_rows = []
for condition in ['healthy', 'aki']:
    condition_mask = adata_pt.obs['comparison_condition'].eq(condition).to_numpy()
    condition_pseudospace = pooled_pseudospace[condition_mask]
    condition_design = make_gam_design(condition_pseudospace, pathway_gam_knots)
    _, raw_beta, _ = dense_gam_sse(
        raw_top_pathway_scores[condition_mask],
        condition_design,
        single_smooth_slice(condition_design),
    )
    raw_curves = (pathway_grid_design @ raw_beta).T
    for curve_i, row in enumerate(top_pathways.itertuples(index=False)):
        for x, value in zip(grid, raw_curves[curve_i]):
            raw_curve_rows.append({
                'library': row.library, 'pathway': row.pathway, 'condition': condition,
                'pseudospace': x, 'fitted_raw_log_normalized_expression': value,
            })
top_pathway_raw_lognorm_curves = pd.DataFrame(raw_curve_rows)
top_pathway_raw_lognorm_curves.to_csv(OUTPUT_DIR / 'top_pathway_raw_lognorm_curves.csv', index=False)

ncols = 3
nrows = max(1, int(np.ceil(len(top_pathways) / ncols)))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.4 * nrows), sharex=True)
axes = np.atleast_1d(axes).ravel()
for ax, row in zip(axes, top_pathways.itertuples(index=False)):
    subset = top_pathway_raw_lognorm_curves.loc[
        top_pathway_raw_lognorm_curves['library'].eq(row.library)
        & top_pathway_raw_lognorm_curves['pathway'].eq(row.pathway)
    ]
    sns.lineplot(
        data=subset, x='pseudospace', y='fitted_raw_log_normalized_expression',
        hue='condition', ax=ax, linewidth=2,
    )
    ax.set_title(f'{row.library}: {row.pathway}', fontsize=9)
    ax.set_xlabel('Shared pseudospace')
    ax.set_ylabel('Mean raw log-normalized expression')
for ax in axes[len(top_pathways):]:
    ax.axis('off')
fig.suptitle('I2. Top prioritized pathway trajectories, raw log-normalized expression', y=1.01)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'I2_top_pathway_raw_lognorm_trajectories.png', bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(OUTPUT_DIR / 'I2_top_pathway_raw_lognorm_trajectories.png')))
display(top_pathway_raw_lognorm_curves.head())


### Per-Sample Predefined Pathway Curves

Plots one per-sample sensitivity figure for selected predefined pathway terms.


In [ ]:
# Figure J: per-sample curves for top effect-size-ranked pathways.
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.4 * nrows), sharex=True)
axes = np.atleast_1d(axes).ravel()
for ax, row in zip(axes, top_pathways.itertuples(index=False)):
    subset = top_sample_pathway_curves.loc[
        top_sample_pathway_curves['library'].eq(row.library)
        & top_sample_pathway_curves['pathway'].eq(row.pathway)
    ]
    sns.lineplot(
        data=subset, x='pseudospace', y='fitted_pathway_score', hue='sample',
        style='condition', ax=ax, linewidth=2,
    )
    ax.set_title(f'{row.library}: {row.pathway}', fontsize=9)
    ax.set_xlabel('Shared pseudospace')
    ax.set_ylabel('Mean gene z-score')
for ax in axes[len(top_pathways):]:
    ax.axis('off')
fig.suptitle('J. Per-sample curves for top prioritized pathways', y=1.01)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'J_top_pathway_per_sample_curves.png', bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(OUTPUT_DIR / 'J_top_pathway_per_sample_curves.png')))

## Cross-Group Profile Similarity and Visualization

Equal-width bins are used only for descriptive profile-similarity summaries after the
sample-aware GAMs have been fitted. Segment-wise profile similarity is quantified
using genes classified as dynamic in at least one condition. For each of the `30` shared
pseudospace bins, the analysis correlates
the healthy and AKI vectors of mean log-normalized expression across this dynamic-gene
set. Spearman correlation between matched healthy and AKI bin positions is reported as
the segment-wise profile similarity score. The full `30 x 30` cross-bin matrix is also
calculated to show agreement between every healthy-bin and AKI-bin combination.

Bins without sufficient finite values are retained as missing values. No interpolation
is applied. The matched-position scores and full cross-bin matrix are written to
`segmentwise_profile_similarity_score.csv` and
`cross_condition_bin_correlation_matrix.csv`.

For the paired dynamic-gene heatmap, genes dynamic in both conditions are ranked by
healthy-AKI fitted-curve correlation. The first `60` genes are displayed after
within-gene, within-condition z-score standardization. If no genes are dynamic in both
conditions, the fallback set is genes dynamic in either condition.

The following output figures are generated:

- `A_canonical_marker_gradients.png`: canonical S1-S3 marker trajectories.
- `B_paired_dynamic_ortholog_heatmap.png`: paired healthy and AKI dynamic-gene heatmaps.
- `C_peak_shift_scatter.png`: healthy versus AKI fitted peak positions, colored by curve correlation and sized by amplitude ratio.
- `F_per_sample_marker_curves.png`: sample-specific curves for every retained canonical marker.
- `mouse_healthy_vs_aki_PT_summary_figure.png`: combined manuscript-oriented overview.


Each saved PNG is displayed inline immediately after it is written so that the executed
notebook contains the complete visual output without requiring separate file browsing.

### Cross-Group Profile Similarity and Heatmap Inputs

Computes cross-condition bin-correlation summaries and selects genes with dynamic signal detected in both groups for the paired heatmap.


In [ ]:
# Correlate healthy and AKI binned profiles using genes dynamic in at least one condition.
dynamic_any = healthy['dynamic'] | aki['dynamic']
if dynamic_any.sum() < 20:
    dynamic_any = np.ones(len(genes), dtype=bool)
healthy_bins = healthy['binned_means'][dynamic_any]
aki_bins = aki['binned_means'][dynamic_any]
bin_centers = healthy['centers']
profile_similarity_matrix = np.full((CONFIG['n_bins'], CONFIG['n_bins']), np.nan)
for i in range(CONFIG['n_bins']):
    for j in range(CONFIG['n_bins']):
        valid = np.isfinite(healthy_bins[:, i]) & np.isfinite(aki_bins[:, j])
        if valid.sum() >= 20:
            profile_similarity_matrix[i, j] = safe_spearman(healthy_bins[valid, i], aki_bins[valid, j])
diagonal_profile_similarity = np.diag(profile_similarity_matrix)
profile_similarity_score = pd.DataFrame({
    'pseudospace': bin_centers,
    'healthy_aki_profile_spearman': diagonal_profile_similarity,
})
profile_similarity_score.to_csv(OUTPUT_DIR / 'segmentwise_profile_similarity_score.csv', index=False)
pd.DataFrame(
    profile_similarity_matrix, index=bin_centers, columns=bin_centers
).to_csv(OUTPUT_DIR / 'cross_condition_bin_correlation_matrix.csv')

# Select genes dynamic in both groups for the paired heatmap, prioritizing high shape correlation.
both_dynamic = healthy['dynamic'] & aki['dynamic']
heatmap_order = np.where(both_dynamic)[0]
if heatmap_order.size == 0:
    heatmap_order = np.where(dynamic_any)[0]
heatmap_rank_values = np.nan_to_num(curve_corr[heatmap_order], nan=-np.inf)
heatmap_order = heatmap_order[np.argsort(heatmap_rank_values)[::-1]]
heatmap_order = heatmap_order[:CONFIG['heatmap_genes']]
healthy_heat = zscore_rows(healthy['curves'][heatmap_order])
aki_heat = zscore_rows(aki['curves'][heatmap_order])
heatmap_genes = genes[heatmap_order]
heatmap_table = pd.concat([
    pd.DataFrame(healthy_heat, index=heatmap_genes, columns=[f'{x:.2f}' for x in grid]).assign(condition='healthy'),
    pd.DataFrame(aki_heat, index=heatmap_genes, columns=[f'{x:.2f}' for x in grid]).assign(condition='aki'),
])
heatmap_table.to_csv(OUTPUT_DIR / 'paired_dynamic_heatmap_values.csv')
display(profile_similarity_score)
display(pd.DataFrame({'heatmap_gene': heatmap_genes}).head(20))

### Canonical Marker Gradient Figure

Plots canonical proximal-tubule marker gradients to validate the shared pseudospace axis.


In [ ]:
# Figure A: canonical marker-gradient validation.
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
for ax, group in zip(axes, MARKER_GROUPS):
    subset = marker_curves.loc[marker_curves['marker_group'].eq(group)]
    sns.lineplot(
        data=subset, x='pseudospace', y='fitted_expression', hue='gene',
        style='condition', ax=ax, linewidth=2,
    )
    ax.set_title(f'{group} markers')
    ax.set_xlabel('Shared pseudospace')
axes[0].set_ylabel('GAM-fitted log-normalized expression')
fig.suptitle('A. Descriptive PT marker gradients in healthy mouse and AKI tubules', y=1.03)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'A_canonical_marker_gradients.png', bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(OUTPUT_DIR / 'A_canonical_marker_gradients.png')))

### Paired Dynamic Ortholog Heatmap

Plots paired healthy and AKI heatmaps for the selected genes with dynamic signal detected in both groups and similar fitted shapes.


In [ ]:
# Figure B: paired dynamic-gene heatmaps.
fig, axes = plt.subplots(1, 2, figsize=(11, max(5, 0.16 * len(heatmap_genes))), sharey=True)
for ax, values, title in zip(axes, [healthy_heat, aki_heat], ['Healthy', 'AKI']):
    sns.heatmap(values, cmap='vlag', center=0, vmin=-2.5, vmax=2.5, yticklabels=heatmap_genes, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Shared pseudospace bins')
axes[0].set_ylabel('Dynamic in both; similar fitted shape')
fig.suptitle('B. Dynamic in both; similar fitted shape trajectories', y=1.01)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'B_paired_dynamic_ortholog_heatmap.png', bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(OUTPUT_DIR / 'B_paired_dynamic_ortholog_heatmap.png')))

### Ortholog Peak-Shift Scatter

Plots healthy versus aki peak locations for dynamic ortholog trajectories.


In [ ]:
# Figure C: peak-shift scatter.
plot_stats = ortholog_stats.loc[ortholog_stats['healthy_dynamic'] | ortholog_stats['aki_dynamic']].copy()
fig, ax = plt.subplots(figsize=(6.5, 6))
points = ax.scatter(
    plot_stats['healthy_peak_pseudospace'], plot_stats['aki_peak_pseudospace'],
    c=plot_stats['curve_spearman'], s=16 + 55 * np.clip(plot_stats['amplitude_ratio'], 1, 4) / 4,
    cmap='coolwarm', vmin=-1, vmax=1, alpha=0.75, edgecolor='none',
)
ax.plot([0, 1], [0, 1], color='black', linestyle='--', linewidth=1)
ax.set(xlabel='Healthy peak pseudospace', ylabel='AKI peak pseudospace',
       title='C. Peak shifts among dynamic orthologs')
fig.colorbar(points, ax=ax, label='Curve Spearman correlation')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'C_peak_shift_scatter.png', bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(OUTPUT_DIR / 'C_peak_shift_scatter.png')))

### Per-Sample Marker Curves

Plots per-sample canonical marker GAM curves to expose sample-level sensitivity.


In [ ]:
# Figure F: sample-specific canonical-marker trajectories.
available_markers = sample_marker_curves[['marker_group', 'gene']].drop_duplicates()
ncols = 3
nrows = int(np.ceil(len(available_markers) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.2 * nrows), sharex=True)
axes = np.atleast_1d(axes).ravel()
for ax, (_, marker) in zip(axes, available_markers.iterrows()):
    subset = sample_marker_curves.loc[sample_marker_curves['gene'].eq(marker['gene'])]
    sns.lineplot(
        data=subset, x='pseudospace', y='fitted_expression', hue='sample',
        style='condition', ax=ax, linewidth=2,
    )
    ax.set_title(f"{marker['marker_group']}: {marker['gene']}")
    ax.set_xlabel('Shared pseudospace')
    ax.set_ylabel('GAM-fitted expression')
for ax in axes[len(available_markers):]:
    ax.axis('off')
fig.suptitle('F. Per-sample canonical PT marker trajectories', y=1.01)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'F_per_sample_marker_curves.png', bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(OUTPUT_DIR / 'F_per_sample_marker_curves.png')))

### Saved Output Summary

Displays the segment-wise profile similarity table and the output directory for generated figures and tables.


In [ ]:
print('\nSegment-wise profile similarity score:')
print(profile_similarity_score.to_string(index=False))
print(f'\nSaved tables and figures to: {OUTPUT_DIR.relative_to(PROJECT_DIR)}')

## Interpretation notes

- `ortholog_trajectory_classification.csv` contains blocked permutation p-values/FDR,
  fitted trajectories, and direct curve-difference effect sizes.
- `top_gene_trajectory_statistics.csv`, group-level curves, and per-sample curves provide
  effect-size-ranked descriptive prioritization among statistically dynamic genes.
- `functional_enrichment_all_terms.csv` and
  `functional_enrichment_significant_terms.csv` contain enrichment p-values/FDR and
  enrichment-fold effect sizes.
- `predefined_pathways_for_gam_modeling.csv` records the independently defined pathway
  panel and attached enrichment results.
- `pathway_trajectory_classification.csv` contains blocked permutation p-values/FDR,
  effect sizes, correlations, peaks, and trajectory classes for the predefined panel.
- Canonical marker, profile-similarity, heatmap, and per-sample figures remain descriptive
  checks of trajectory structure and sample consistency.

Permutation and enrichment p-values determine statistical significance. Effect sizes rank
the significant results. Healthy-versus-AKI differences remain descriptive until more
independent biological samples are available.
